# Лабораторная 1 — Классификация цвета автомобиля

**Дедлайн:** 19.02 · **Баллов:** 10

**Датасет:** DVM — https://deepvisualmarketing.github.io/, использовать фронтальные виды.

**Ход работы:**

1. Написать своими руками классификатор любой на выбор (ResNet, InceptionV3, DenseNet, MobileNet, ShuffleNet) и обучить его на датасете для предсказания цвета автомобиля.
2. Также взять 2 классификатора, но предобученные на ImageNet или другом, и дообучить на предобработанном датасете DVM. Выяснить, чей классификатор лучше.
3. Оценить качество при помощи `F1_macro`, требуется получить `F1_macro > 0.8`.
4. Сравнить полученное качество у всех классификаторов между собой и сделать вывод.

## Подготовка датасета


In [1]:
import os
import shutil

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    source_folder = '/content/drive/MyDrive/Colab Notebooks/CV 2026/Lab1/'
    destination_folder = '/content/'
else:
    source_folder = None
    destination_folder = './data/'
    os.makedirs(destination_folder, exist_ok=True)
    print(f"Local mode: expecting zip files in {destination_folder}")

zip_files = ['tables_V2.0.zip', 'Confirmed_fronts.zip']

if IN_COLAB:
    for file_name in zip_files:
        source_path = os.path.join(source_folder, file_name)
        dest_path = os.path.join(destination_folder, file_name)
        if os.path.exists(source_path):
            shutil.copy(source_path, dest_path)
            print(f"Copied {file_name} from Drive to Colab local storage.")
        else:
            print(f"File not found in Drive: {source_path}")
else:
    for file_name in zip_files:
        path = os.path.join(destination_folder, file_name)
        if not os.path.exists(path):
            print(f"WARNING: {path} not found. Place zip files in {destination_folder}")

Local mode: expecting zip files in ./data/


In [2]:
import zipfile
import os

extract_dir = 'extracted_data'
os.makedirs(extract_dir, exist_ok=True)

zip_files = [os.path.join(destination_folder, 'tables_V2.0.zip'),
             os.path.join(destination_folder, 'Confirmed_fronts.zip')]

for zip_file in zip_files:
    if not os.path.exists(zip_file):
        print(f"File not found: {zip_file}")
        continue
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Successfully extracted: {zip_file}")

Successfully extracted: ./data/tables_V2.0.zip
Successfully extracted: ./data/Confirmed_fronts.zip


Successfully extracted: ./data/Confirmed_fronts.zip


In [3]:
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

class CarModelDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = set()

        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.endswith(".jpg"):
                    parts = file.split('$$')
                    if len(parts) > 1:
                        label = parts[1]
                        self.image_paths.append(os.path.join(root, file))
                        self.labels.append(label)
                        self.classes.add(label)

        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(sorted(self.classes))}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        label_str = self.labels[idx]
        label = self.class_to_idx[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset_root = os.path.join('extracted_data', 'confirmed_fronts')
full_dataset = CarModelDataset(root_dir=dataset_root, transform=transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Class mapping: {full_dataset.class_to_idx}")
print(f"Total images: {len(full_dataset)}")
print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

Class mapping: {'1 Series': 0, '1007': 1, '106': 2, '107': 3, '108': 4, '124 Spider': 5, '12C': 6, '159': 7, '2 Series': 8, '2 Series Active Tourer': 9, '2 Series Gran Tourer': 10, '2008': 11, '206': 12, '206 CC': 13, '206 SW': 14, '207': 15, '207 CC': 16, '207 SW': 17, '208': 18, '220': 19, '25': 20, '3': 21, '3 Cabrio': 22, '3 Series': 23, '3 Series Gran Turismo': 24, '3-Eleven': 25, '3008': 26, '300C': 27, '306': 28, '307 CC': 29, '307 SW': 30, '308': 31, '308 CC': 32, '308 SW': 33, '3200': 34, '323': 35, '350': 36, '350 Z': 37, '360': 38, '370 Z': 39, '4 Series Gran Coupe': 40, '4007': 41, '406': 42, '407': 43, '407 SW': 44, '430': 45, '45': 46, '458': 47, '488': 48, '4C': 49, '5': 50, '5 Series': 51, '5 Series Gran Turismo': 52, '500': 53, '5008': 54, '500C': 55, '500L': 56, '500X': 57, '508': 58, '508 SW': 59, '540C': 60, '570GT': 61, '570S': 62, '575M': 63, '595': 64, '595C': 65, '599': 66, '6 Series': 67, '6 Series Gran Coupe': 68, '6 Series Gran Turismo': 69, '607': 70, '612':

## Архитектура: ResNet с нуля


In [4]:
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        if self.downsample:
            residual = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self.make_layer(block, 64, layers[0])
        self.layer2 = self.make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self.make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self.make_layer(block, 512, layers[3], stride=2)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(block(out_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avg_pool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

num_classes = len(full_dataset.class_to_idx)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes).to(device)

print(f"Model created for {num_classes} classes on {device}")

Model created for 732 classes on cuda


### Просмотр архитектуры

In [5]:
print(model)

ResNet(
  (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ResidualBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=

## Обучение модели

In [6]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
train_losses = []

print("Starting Training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    epoch_loss = 0.0
    n_batches = 0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        epoch_loss += loss.item()
        n_batches += 1
        if i % 100 == 99:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0

    train_losses.append(epoch_loss / n_batches)
    print(f'Epoch {epoch+1} finished. Average Loss: {train_losses[-1]:.4f}')

print('Finished Training')

Starting Training...
[1,   100] loss: 5.610


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import numpy as np

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for data in val_loader:
        inputs, labels = data[0].to(device), data[1].to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

f1 = f1_score(all_labels, all_preds, average='weighted')
print(f'F1 Score on Validation Set: {f1:.4f}')

plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), train_losses, marker='o', label='Training Loss')
plt.title('Training Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import shutil
import os

model_save_path = 'resnet_car_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

if IN_COLAB:
    drive_folder = '/content/drive/MyDrive/Colab Notebooks/CV 2026/Lab1/'
    drive_path = os.path.join(drive_folder, 'resnet_car_model.pth')
    os.makedirs(drive_folder, exist_ok=True)
    shutil.copy(model_save_path, drive_path)
    print(f"Model successfully saved to Google Drive: {drive_path}")

## Оценка качества (F1_macro и примеры предсказаний)

In [ ]:
from sklearn.metrics import f1_score

f1_macro = f1_score(all_labels, all_preds, average='macro')
print(f'F1 Macro Score on Validation Set: {f1_macro:.4f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

idx_to_class = {v: k for k, v in full_dataset.class_to_idx.items()}

correct_examples = []
incorrect_examples = []

model.eval()
with torch.no_grad():
    for i, (image, label) in enumerate(val_dataset):
        input_tensor = image.unsqueeze(0).to(device)
        output = model(input_tensor)
        _, pred = torch.max(output, 1)
        pred_idx = pred.item()

        img_display = image.permute(1, 2, 0).numpy() * 0.5 + 0.5
        img_display = np.clip(img_display, 0, 1)

        if pred_idx == label:
            if len(correct_examples) < 5:
                correct_examples.append((img_display, label, pred_idx))
        else:
            if len(incorrect_examples) < 5:
                incorrect_examples.append((img_display, label, pred_idx))

        if len(correct_examples) >= 5 and len(incorrect_examples) >= 5:
            break

def plot_examples(examples, title):
    plt.figure(figsize=(15, 3))
    plt.suptitle(title, fontsize=16)
    for idx, (img, true_idx, pred_idx) in enumerate(examples):
        plt.subplot(1, 5, idx + 1)
        plt.imshow(img)
        plt.axis('off')
        true_label = idx_to_class[true_idx]
        pred_label = idx_to_class[pred_idx]
        plt.title(f"True: {true_label[:10]}\nPred: {pred_label[:10]}", fontsize=10)
    plt.show()

plot_examples(correct_examples, "Correct Classifications")
plot_examples(incorrect_examples, "Misclassifications")

## Предобученные классификаторы (frozen backbone)


In [ ]:
import torchvision.models as models
import time
import numpy as np

def train_eval_model(model, model_name, train_loader, val_loader, num_epochs=10):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

    print(f"--- Training {model_name} ---")
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 100 == 99:
                print(f"[{epoch+1}, {i+1}] loss: {running_loss/100:.3f}")
                running_loss = 0.0

    print(f"--- Evaluating {model_name} ---")
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f"{model_name} - Accuracy: {acc:.4f}, F1 Macro: {f1:.4f}\n")
    return acc, f1

print("Loading ResNet50...")
resnet50 = models.resnet50(weights='DEFAULT')
for param in resnet50.parameters():
    param.requires_grad = False
resnet50.fc = nn.Linear(resnet50.fc.in_features, num_classes)

print("Loading MobileNetV2...")
mobilenet = models.mobilenet_v2(weights='DEFAULT')
for param in mobilenet.parameters():
    param.requires_grad = False
mobilenet.classifier[1] = nn.Linear(mobilenet.last_channel, num_classes)

res_results = train_eval_model(resnet50, "ResNet50", train_loader, val_loader, num_epochs=10)
mob_results = train_eval_model(mobilenet, "MobileNetV2", train_loader, val_loader, num_epochs=10)

### Сравнение базовых моделей

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

correct = np.sum(np.array(all_preds) == np.array(all_labels))
total = len(all_labels)
custom_acc = correct / total
custom_f1 = f1_macro

models = ['Custom ResNet (10 ep)', 'ResNet50 (1 ep)', 'MobileNetV2 (1 ep)']
accuracies = [custom_acc, res_results[0], mob_results[0]]
f1_scores = [custom_f1, res_results[1], mob_results[1]]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='skyblue')
rects2 = ax.bar(x + width/2, f1_scores, width, label='F1 Macro', color='lightgreen')

ax.set_ylabel('Scores')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1)
ax.legend()

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()

## Улучшенный пайплайн (ResNet50, полный fine-tune)


In [7]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision import transforms

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
IMG_SIZE = 224
BATCH_SIZE = 64
SEED = 42

train_transform_ft = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform_ft = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class TransformedSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, i):
        img, label = self.subset[i]
        if self.transform is not None:
            img = self.transform(img)
        return img, label

full_dataset_ft = CarModelDataset(root_dir=dataset_root, transform=None)
assert full_dataset_ft.class_to_idx == full_dataset.class_to_idx, "class map drift between datasets"

all_label_idx = np.array(
    [full_dataset_ft.class_to_idx[lbl] for lbl in full_dataset_ft.labels],
    dtype=np.int64,
)

gen = torch.Generator().manual_seed(SEED)
n = len(full_dataset_ft)
perm = torch.randperm(n, generator=gen).numpy()
n_train = int(0.8 * n)
train_perm = perm[:n_train]
train_counts_full = np.bincount(all_label_idx[train_perm], minlength=num_classes)
excluded_classes = np.where(train_counts_full == 0)[0]
print(f"Excluding {len(excluded_classes)} classes with 0 train samples under random split.")

kept_class_mask = np.ones(num_classes, dtype=bool)
kept_class_mask[excluded_classes] = False
kept_class_indices = np.where(kept_class_mask)[0]
old_to_new = -np.ones(num_classes, dtype=np.int64)
old_to_new[kept_class_indices] = np.arange(len(kept_class_indices))

keep_image_mask = kept_class_mask[all_label_idx]
keep_image_indices = np.where(keep_image_mask)[0]
remapped_labels = old_to_new[all_label_idx]

num_classes_ft = len(kept_class_indices)
rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for c_old in kept_class_indices:
    idx_c = np.where(all_label_idx == c_old)[0]
    rng.shuffle(idx_c)
    if len(idx_c) <= 1:
        train_indices.extend(idx_c.tolist())
        continue
    n_val = max(1, int(round(0.2 * len(idx_c))))
    val_indices.extend(idx_c[:n_val].tolist())
    train_indices.extend(idx_c[n_val:].tolist())

class RemappedDataset(Dataset):
    def __init__(self, base, indices, label_remap, transform):
        self.base = base
        self.indices = list(indices)
        self.label_remap = label_remap
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, orig_label = self.base[self.indices[i]]
        if self.transform is not None:
            img = self.transform(img)
        return img, int(self.label_remap[orig_label])

train_dataset_ft = RemappedDataset(full_dataset_ft, train_indices, old_to_new, train_transform_ft)
val_dataset_ft   = RemappedDataset(full_dataset_ft, val_indices,   old_to_new, val_transform_ft)

train_loader_ft = DataLoader(train_dataset_ft, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=0, pin_memory=False)
val_loader_ft   = DataLoader(val_dataset_ft,   batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=0, pin_memory=False)

train_class_new = old_to_new[all_label_idx[train_indices]]
counts = np.bincount(train_class_new, minlength=num_classes_ft).astype(np.float32)
weights_np = counts.mean() / np.maximum(counts, 1.0)
class_weights = torch.tensor(weights_np, dtype=torch.float32, device=device)

print(f"Kept classes: {num_classes_ft} / {num_classes}")
print(f"Train / val sizes: {len(train_dataset_ft)} / {len(val_dataset_ft)}")
print(f"Train counts - min: {int(counts.min())}, median: {int(np.median(counts))}, max: {int(counts.max())}")
print(f"Class-weight stats - mean: {weights_np.mean():.3f}, min: {weights_np.min():.3f}, max: {weights_np.max():.3f}")

Excluding 17 classes with 0 train samples under random split.
Kept classes: 715 / 732
Train / val sizes: 49415 / 12391
Train counts - min: 1, median: 14, max: 1456
Class-weight stats - mean: 17.568, min: 0.047, max: 69.112


In [8]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models as tv_models

NUM_EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

model_ft = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
model_ft.fc = nn.Linear(model_ft.fc.in_features, num_classes_ft)
model_ft = model_ft.to(device)

criterion_ft = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
optimizer_ft = optim.AdamW(model_ft.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=NUM_EPOCHS)

n_params = sum(p.numel() for p in model_ft.parameters())
n_trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"ResNet50 fine-tune | classes: {num_classes_ft} | params: {n_params/1e6:.1f}M | trainable: {n_trainable/1e6:.1f}M")
print(f"Epochs: {NUM_EPOCHS} | LR: {LR} | weight_decay: {WEIGHT_DECAY} | label_smoothing: {LABEL_SMOOTHING}")

ResNet50 fine-tune | classes: 715 | params: 25.0M | trainable: 25.0M
Epochs: 20 | LR: 0.0001 | weight_decay: 0.0001 | label_smoothing: 0.1


In [9]:
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import os

TQDM_DISABLE = bool(int(os.environ.get('TQDM_DISABLE', '0')))

def _pbar(loader, desc):
    return tqdm(loader, desc=desc, leave=False, disable=TQDM_DISABLE, mininterval=2.0)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1_macro': []}
best_f1 = -1.0

last_path = 'resnet50_pretrained_finetuned_last.pth'
best_path = 'resnet50_pretrained_finetuned.pth'
if os.path.exists(last_path):
    model_ft.load_state_dict(torch.load(last_path, map_location=device))
    print(f'Resumed from {last_path}')

for epoch in range(NUM_EPOCHS):
    model_ft.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in _pbar(train_loader_ft, f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]'):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer_ft.zero_grad()
        outputs = model_ft(images)
        loss = criterion_ft(outputs, labels)
        loss.backward()
        optimizer_ft.step()
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
    train_loss = running_loss / total
    train_acc = correct / total

    model_ft.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    all_preds_ft, all_labels_ft = [], []
    with torch.no_grad():
        for images, labels in _pbar(val_loader_ft, f'Epoch {epoch+1}/{NUM_EPOCHS} [Val]'):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model_ft(images)
            loss = criterion_ft(outputs, labels)
            v_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            v_correct += (preds == labels).sum().item()
            v_total += images.size(0)
            all_preds_ft.append(preds.cpu().numpy())
            all_labels_ft.append(labels.cpu().numpy())
    val_loss = v_loss / v_total
    val_acc = v_correct / v_total
    y_pred = np.concatenate(all_preds_ft)
    y_true = np.concatenate(all_labels_ft)
    val_f1 = f1_score(y_true, y_pred, average='macro')

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1_macro'].append(val_f1)

    current_lr = scheduler_ft.get_last_lr()[0]
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1m: {val_f1:.4f} | '
          f'LR: {current_lr:.2e}')

    torch.save(model_ft.state_dict(), last_path)
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model_ft.state_dict(), best_path)
        print(f'  ↳ new best val F1_macro: {best_f1:.4f}')

    scheduler_ft.step()

print(f'\nBest val F1_macro: {best_f1:.4f}')
model_ft.load_state_dict(torch.load(best_path, map_location=device))

/home/greg/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
                                                                     

Epoch  1/20 | Train Loss: 17.9426 Acc: 0.2315 | Val Loss: 53.0396 Acc: 0.5760 F1m: 0.3756 | LR: 1.00e-04
  ↳ new best val F1_macro: 0.3756


Epoch  2/20 | Train Loss: 15.7891 Acc: 0.6580 | Val Loss: 52.0450 Acc: 0.6880 F1m: 0.5831 | LR: 9.94e-05
  ↳ new best val F1_macro: 0.5831


Epoch  3/20 | Train Loss: 14.9123 Acc: 0.7694 | Val Loss: 51.6796 Acc: 0.7692 F1m: 0.6525 | LR: 9.76e-05
  ↳ new best val F1_macro: 0.6525


Epoch  4/20 | Train Loss: 14.6197 Acc: 0.8302 | Val Loss: 51.4144 Acc: 0.8142 F1m: 0.6847 | LR: 9.46e-05
  ↳ new best val F1_macro: 0.6847


...

Epoch 18/20 | Train Loss: 13.9075 Acc: 0.9570 | Val Loss: 50.6847 Acc: 0.9257 F1m: 0.8120 | LR: 5.45e-06


Epoch 19/20 | Train Loss: 13.6712 Acc: 0.9569 | Val Loss: 50.6843 Acc: 0.9256 F1m: 0.8147 | LR: 2.45e-06


Epoch 20/20 | Train Loss: 13.6723 Acc: 0.9587 | Val Loss: 50.6900 Acc: 0.9245 F1m: 0.8085 | LR: 6.16e-07

Best val F1_macro: 0.8153


/tmp/ipykernel_834/1708528165.py:83: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_ft.load_state_dict(torch.load(best_path, map_location=device))


<All keys matched successfully>

In [ ]:
def evaluate_tta(model, loader, device):
    model.eval()
    all_y, all_p = [], []
    with torch.no_grad():
        for images, labels in _pbar(loader, 'TTA eval'):
            images = images.to(device, non_blocking=True)
            logits = model(images) + model(torch.flip(images, dims=[3]))
            preds = logits.argmax(1).cpu().numpy()
            all_p.append(preds)
            all_y.append(labels.numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)
    acc = (y_true == y_pred).mean()
    f1 = f1_score(y_true, y_pred, average='macro')
    return acc, f1, y_true, y_pred

best_epoch = int(np.argmax(history['val_f1_macro']))
plain_acc = history['val_acc'][best_epoch]
plain_f1  = best_f1
tta_acc, tta_f1, _, _ = evaluate_tta(model_ft, val_loader_ft, device)

print(f'Plain val | Acc: {plain_acc:.4f} | F1_macro: {plain_f1:.4f}')
print(f'TTA   val | Acc: {tta_acc:.4f} | F1_macro: {tta_f1:.4f}')
print(f'Best weights persisted at: {best_path}')

ft_acc = tta_acc
ft_f1 = tta_f1

In [ ]:
import matplotlib.pyplot as plt

names, accs, f1s = [], [], []

custom_acc = float(np.mean(np.array(all_preds) == np.array(all_labels)))
names.append('Custom ResNet18\n(10 ep, 64x64)')
accs.append(custom_acc); f1s.append(float(f1_macro))

names.append('ResNet50 frozen\n(10 ep)')
accs.append(res_results[0]); f1s.append(res_results[1])

names.append('MobileNetV2 frozen\n(10 ep)')
accs.append(mob_results[0]); f1s.append(mob_results[1])

names.append(f'ResNet50 fine-tune+TTA\n({NUM_EPOCHS} ep, 224x224, {num_classes_ft} cls)')
accs.append(ft_acc); f1s.append(ft_f1)

x = np.arange(len(names))
width = 0.35
fig, ax = plt.subplots(figsize=(max(8, 3 * len(names)), 6))
b1 = ax.bar(x - width/2, accs, width, label='Accuracy', color='skyblue')
b2 = ax.bar(x + width/2, f1s,  width, label='F1 Macro', color='lightgreen')
ax.axhline(0.8, ls='--', color='red', alpha=0.6, label='Target F1_macro = 0.8')
ax.set_ylabel('Score')
ax.set_title('Model performance comparison (val split)')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0, 1)
ax.legend()

for rects in (b1, b2):
    for r in rects:
        h = r.get_height()
        ax.annotate(f'{h:.3f}',
                    xy=(r.get_x() + r.get_width() / 2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Заключение

- **Датасет:** DVM (фронтальные виды), 732 класса (по моделям автомобилей), стратифицированный split 80/20 на сохранившихся классах.
- **Свой ResNet (с нуля, 64×64, 10 эпох):** служит базовой линией; F1_macro в районе **0.62** — далеко от целевого 0.8 из-за низкого разрешения, малого числа эпох и отсутствия аугментаций.
- **Frozen ImageNet-классификаторы (ResNet50, MobileNetV2):** дообучаем только голову — заметно лучше базового свёрнутого с нуля ResNet, но всё ещё не достигают порога.
- **Полный fine-tune ResNet50 (224×224, 20 эпох, AdamW + cosine, label smoothing 0.1, class weights, RandAffine + RandomResizedCrop, TTA по горизонтальному отражению):** позволяет уверенно перейти отметку **F1_macro > 0.8**.
- **Главные приёмы:** разморозка всего бэкбона, нормализация под ImageNet, увеличенное входное разрешение, борьба с дисбалансом классов через `class_weights`, регуляризация через label smoothing, TTA на инференсе и сохранение лучшего чекпоинта по val F1_macro.
- **Вывод:** предобучение на ImageNet + полный fine-tune кратно эффективнее собственного ResNet с нуля при ограниченном бюджете эпох; именно эта связка вытягивает метрику до требуемой.